In [13]:
from pathlib import Path
import re
import shutil

WDI = Path("data/raw_wsi")
MDI = Path("data/masks")
DRY_RUN = False  # set False when you're happy

In [14]:
def sanitize_stem(stem: str) -> str:
    # strip
    s = stem.strip()
    # replace spaces with underscore
    s = re.sub(r"\s+", "_", s)
    # replace ':' with '-'
    s = s.replace(":", "-")
    # timestamps often have dots -> safer to replace with '-'
    # but keep extension dots separate (we only touch stem)
    s = s.replace(".", "-")
    # remove double separators
    s = re.sub(r"_+", "_", s)
    s = re.sub(r"-+", "-", s)
    # remove trailing underscores/dashes
    s = s.strip("_-")
    return s

In [15]:
wsi_exts = {".svs", ".ndpi", ".mrxs", ".tif", ".tiff"}
wsi_files = [p for p in WDI.iterdir() if p.suffix.lower() in wsi_exts]

plan = []
for p in sorted(wsi_files):
    new_stem = sanitize_stem(p.stem)
    new_p = p.with_name(new_stem + p.suffix.lower())
    plan.append((p, new_p))

# show changes
for old, new in plan:
    if old.name != new.name:
        print(f"{old.name}  ->  {new.name}")


targets = [new for _, new in plan]
dups = {p.name for p in targets if sum(t.name == p.name for t in targets) > 1}
print("Collisions:", dups)
assert len(dups) == 0, "Resolve collisions before renaming."

for old, new in plan:
    if old == new:
        continue
    if DRY_RUN:
        print("[DRY]", old.name, "->", new.name)
    else:
        old.rename(new)


mask_exts = [".npy", ".png", ".tif", ".tiff"]

# build map from old stem -> new stem based on WSI plan
stem_map = {old.stem: new.stem for old, new in plan}

mask_files = [p for p in MDI.iterdir() if p.suffix.lower() in mask_exts]
renamed = 0

for mp in sorted(mask_files):
    if mp.stem in stem_map:
        new_stem = stem_map[mp.stem]
        new_mp = mp.with_name(new_stem + mp.suffix.lower())
        if mp.name != new_mp.name:
            if DRY_RUN:
                print("[DRY]", mp.name, "->", new_mp.name)
            else:
                mp.rename(new_mp)
            renamed += 1

print("Mask rename candidates:", renamed)

Collisions: set()
Mask rename candidates: 0


In [16]:
wsi_files2 = [p for p in WDI.iterdir() if p.suffix.lower() in wsi_exts]
mask_files2 = {p.stem for p in MDI.iterdir() if p.suffix.lower() in mask_exts}

missing = []
for p in sorted(wsi_files2):
    if p.stem not in mask_files2:
        missing.append(p.name)

print("WSIs:", len(wsi_files2))
print("Masks:", len(mask_files2))
print("Missing masks for:", missing[:20], ("..." if len(missing)>20 else ""))

WSIs: 12
Masks: 0
Missing masks for: ['001.svs', '002.svs', '02_-_2024-04-11_11-15-33.ndpi', '06_-_2024-04-11_19-38-10.ndpi', '1012868.svs', '1012869.svs', '1027221.svs', '1027222.svs', '1039402.svs', '1039403.svs', 'C3L-00016-21.svs', 'C3L-00016-22.svs'] 


In [17]:
import numpy as np

def estimate_candidates_from_mask(mask_path, stride):
    if mask_path.suffix.lower() == ".npy":
        m = np.load(mask_path) > 0
    else:
        from PIL import Image
        m = np.array(Image.open(mask_path).convert("L")) > 0

    # sample grid count ~ (#mask_pixels / stride^2) * tissue_fraction
    h, w = m.shape[:2]
    tissue_frac = m.mean()
    approx = (h*w*tissue_frac) / (stride*stride)
    return tissue_frac, int(approx)

# try first 3 masks
some_masks = sorted([p for p in MDI.iterdir() if p.suffix.lower() in mask_exts])[:3]
for mp in some_masks:
    tf64, c64 = estimate_candidates_from_mask(mp, 64)
    tf32, c32 = estimate_candidates_from_mask(mp, 32)
    print(mp.name, "tissue_frac=", round(tf64,3), "approx_cand_64=", c64, "approx_cand_32=", c32)